# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [1]:
class matrix:

    def __init__(self, n_or_data, m=None):
        if m is not None:
            n = n_or_data
            self.data = [[0.0] * m for _ in range(n)]
        else:
            data = n_or_data
            if not isinstance(data, list) or not all(isinstance(row, list) for row in data):
                raise ValueError("Input must be a list of lists.")
            row_lengths = [len(row) for row in data]
            if len(set(row_lengths)) != 1:
                raise ValueError(f"All rows must have the same length. Got: {row_lengths}")
            self.data = [row[:] for row in data]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row_key, col_key = key
            rows = self.data[row_key]
            if isinstance(row_key, slice):
                return matrix([r[col_key] if isinstance(col_key, slice)
                               else [r[col_key]] for r in rows])
            else:
                return rows[col_key]
        elif isinstance(key, slice):
            return matrix(self.data[key])
        else:
            return self.data[key]

    def assign(self, other):
        if isinstance(other, matrix):
            if other.shape() != self.shape():
                raise ValueError(f"Shape mismatch: {other.shape()} vs {self.shape()}")
            self.data = [row[:] for row in other.data]
        elif isinstance(other, list):
            temp = matrix(other)
            if temp.shape() != self.shape():
                raise ValueError(f"Shape mismatch: {temp.shape()} vs {self.shape()}")
            self.data = [row[:] for row in temp.data]
        else:
            raise TypeError("Can only assign from a matrix or a list of lists.")

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        n, m = self.shape()
        return matrix([[self.data[i][j] for i in range(n)] for j in range(m)])

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[self.data[i][n]] for i in range(self.shape()[0])])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([self.data[i][m_0:m_1] for i in range(n_0, n_1)])

    def scalarmul(self, c):
        n, m = self.shape()
        return matrix([[self.data[i][j] * c for j in range(m)] for i in range(n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Cannot add: shapes {self.shape()} and {N.shape()} differ.")
        n, m = self.shape()
        return matrix([[self.data[i][j] + N.data[i][j] for j in range(m)] for i in range(n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Cannot subtract: shapes {self.shape()} and {N.shape()} differ.")
        n, m = self.shape()
        return matrix([[self.data[i][j] - N.data[i][j] for j in range(m)] for i in range(n)])

    def mat_mult(self, N):
        n, k = self.shape()
        k2, p = N.shape()
        if k != k2:
            raise ValueError(f"Cannot multiply: inner dims {k} and {k2} don't match.")
        result = [
            [sum(self.data[i][r] * N.data[r][j] for r in range(k)) for j in range(p)]
            for i in range(n)
        ]
        return matrix(result)

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError(f"Cannot element-multiply: shapes {self.shape()} and {N.shape()} differ.")
        n, m = self.shape()
        return matrix([[self.data[i][j] * N.data[i][j] for j in range(m)] for i in range(n)])

    def equals(self, N):
        if self.shape() != N.shape():
            return False
        n, m = self.shape()
        return all(self.data[i][j] == N.data[i][j] for i in range(n) for j in range(m))

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        return NotImplemented

    def __add__(self, N):
        return self.add(N)

    def __sub__(self, N):
        return self.sub(N)

    def __eq__(self, N):
        if isinstance(N, matrix):
            return self.equals(N)
        return NotImplemented

    def __repr__(self):
        return '\n'.join(str(row) for row in self.data)

In [2]:
# init with n, m
M = matrix(3, 3)
print(M)

# init with list of lists
M2 = matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(M2)

# indexing
print(M2[1][2])   # 6
print(M2[0, 0])   # 1

# assign from matrix
M.assign(M2)
print(M)

# assign from list of lists
M.assign([[9, 8, 7], [6, 5, 4], [3, 2, 1]])
print(M)

# bad input should raise errors
try:
    matrix([[1, 2], [3, 4, 5]])
except ValueError as e:
    print(e)

try:
    matrix(2, 2).assign(M2)
except ValueError as e:
    print(e)

[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]
6
1
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]
[9, 8, 7]
[6, 5, 4]
[3, 2, 1]
All rows must have the same length. Got: [2, 3]
Shape mismatch: (3, 3) vs (2, 2)


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [3]:
M = matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print(M.shape())
print(M.transpose())
print(M.row(1))
print(M.column(2))
print(M.to_list())
print(M.block(0, 2, 1, 3))
print(M[0:2])
print(M[0:2, 1:3])

(3, 3)
[1, 4, 7]
[2, 5, 8]
[3, 6, 9]
[4, 5, 6]
[3]
[6]
[9]
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]
[2, 3]
[5, 6]
[1, 2, 3]
[4, 5, 6]
[2, 3]
[5, 6]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [4]:
def constant(n, m, c):
    return matrix([[float(c)] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

print(constant(2, 3, 7))
print(zeros(3, 2))
print(ones(2, 4))
print(eye(3))

[7.0, 7.0, 7.0]
[7.0, 7.0, 7.0]
[0.0, 0.0]
[0.0, 0.0]
[0.0, 0.0]
[1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.0, 1.0]
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don't forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [5]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print(A.scalarmul(3))
print(A.add(B))
print(A.sub(B))
print(A.mat_mult(B))
print(A.element_mult(B))
print(A.equals(A))
print(A.equals(B))

try:
    A.add(matrix([[1, 2, 3], [4, 5, 6]]))
except ValueError as e:
    print(e)

[3, 6]
[9, 12]
[6, 8]
[10, 12]
[-4, -4]
[-4, -4]
[19, 22]
[43, 50]
[5, 12]
[21, 32]
True
False
Cannot add: shapes (2, 2) and (2, 3) differ.


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


In [6]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print(2 * A)
print(A * 3)
print(A + B)
print(A - B)
print(A * B)
print(A == A)
print(A == B)

[2, 4]
[6, 8]
[3, 6]
[9, 12]
[6, 8]
[10, 12]
[-4, -4]
[-4, -4]
[19, 22]
[43, 50]
True
False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [7]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
C = matrix([[2, 0], [1, 3]])
I = eye(2)

print("(AB)C == A(BC):", ((A * B) * C) == (A * (B * C)))
print("A(B+C) == AB+AC:", (A * (B + C)) == ((A * B) + (A * C)))
print("AB == BA:", (A * B) == (B * A))
print("AI == A:", (A * I) == A)

(AB)C == A(BC): True
A(B+C) == AB+AC: True
AB == BA: False
AI == A: True
